In [ ]:
#desenvolvido por Tatiana Regina

In [ ]:
# Bibliotecas
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt
import numpy as np
import kagglehub

# Download latest version
path = kagglehub.dataset_download("olistbr/brazilian-ecommerce")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Path to dataset files: /kaggle/input/brazilian-ecommerce


In [ ]:
# Carrega os arquivos csv em dataframes pandas:
customers = pd.read_csv(path+"/olist_customers_dataset.csv")
geolocation = pd.read_csv(path+"/olist_geolocation_dataset.csv")
order_items = pd.read_csv(path+"/olist_order_items_dataset.csv")
payments = pd.read_csv(path+"/olist_order_payments_dataset.csv")
reviews = pd.read_csv(path+"/olist_order_reviews_dataset.csv")
orders = pd.read_csv(path+"/olist_orders_dataset.csv")
products = pd.read_csv(path+"/olist_products_dataset.csv")
product_category = pd.read_csv(path+"/product_category_name_translation.csv")

In [ ]:
#Importar tabela de-para entre estados e Região
df_estados_regiao = pd.read_csv("/tb_de_para_estados.csv")
display(df_estados_regiao)

,Sigla,Estado,Região
0,AC,Acre,Norte
1,AP,Amapá,Norte
2,AM,Amazonas,Norte
3,PA,Pará,Norte
4,RO,Rondônia,Norte
5,RR,Roraima,Norte
6,TO,Tocantins,Norte
7,AL,Alagoas,Nordeste
8,BA,Bahia,Nordeste
9,CE,Ceará,Nordeste


In [ ]:
#Agrupando dados de Pedido, Pagamento, Estado de Entrega, Categoria de Produto
#e Avaliações para analise
order_payments = orders.merge(payments, on='order_id')
order_customer = order_payments.merge(customers, on='customer_id')
order_products = order_customer.merge(order_items, on='order_id')
order_all = order_products.merge(products, on='product_id')
order_all = order_all.merge(reviews, on='order_id')


# Convert date columns to datetime objects in the pandas DataFrame
order_all['order_purchase_timestamp'] = pd.to_datetime(order_all['order_purchase_timestamp'])
order_all['order_approved_at'] = pd.to_datetime(order_all['order_approved_at'])
order_all['order_delivered_carrier_date'] = pd.to_datetime(order_all['order_delivered_carrier_date'])
order_all['order_delivered_customer_date'] = pd.to_datetime(order_all['order_delivered_customer_date'])
order_all['order_estimated_delivery_date'] = pd.to_datetime(order_all['order_estimated_delivery_date'])

# Create 'order_date' column by unifying purchase and approved timestamps
# Use order_purchase_timestamp if available, otherwise order_approved_at
order_all['order_date'] = order_all['order_purchase_timestamp'].fillna(order_all['order_approved_at'])

# Criação tempo estimado de entrega: campo dif_deliv_estimat
order_all['dif_deliv_estimat'] = order_all['order_delivered_customer_date'] - order_all['order_estimated_delivery_date']
order_all['dif_deliv_compra'] = order_all['order_delivered_customer_date'] - order_all['order_date']

# Create 'dif_deliv_estimat_days' and 'dif_deliv_compra_days' for Spark compatibility
order_all['dif_deliv_estimat_days'] = order_all['dif_deliv_estimat'].dt.days
order_all['dif_deliv_compra_days'] = order_all['dif_deliv_compra'].dt.days





In [ ]:
##Trazendo de-para de Estados e Regiões para analise
order_all=order_all.left(df_estados_regiao, on=order_all['customer_state']=df_estados_regiao['Sigla'])

In [ ]:
##Validação e avaliação das colunas a serem utilizadas na analise
display(order_all.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117329 entries, 0 to 117328
Data columns (total 42 columns):
 #   Column                         Non-Null Count   Dtype          
---  ------                         --------------   -----          
 0   order_id                       117329 non-null  object         
 1   customer_id                    117329 non-null  object         
 2   order_status                   117329 non-null  object         
 3   order_purchase_timestamp       117329 non-null  datetime64[ns] 
 4   order_approved_at              117314 non-null  datetime64[ns] 
 5   order_delivered_carrier_date   116094 non-null  datetime64[ns] 
 6   order_delivered_customer_date  114858 non-null  datetime64[ns] 
 7   order_estimated_delivery_date  117329 non-null  datetime64[ns] 
 8   payment_sequential             117329 non-null  int64          
 9   payment_type                   117329 non-null  object         
 10  payment_installments           117329 non-null  int64   

None

In [ ]:
##Avaliando os dados na menor granularidade para selecionar apenas as colunas a serem utilizadas na analise dos dados:
display(order_all).where(order_all['order_id']=	'00018f77f2f0320c557190d7a144bdd3')

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,payment_sequential,payment_type,...,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp,order_date,dif_deliv_estimat,dif_deliv_compra,dif_deliv_estimat_days,dif_deliv_compra_days,regiao
84679,00018f77f2f0320c557190d7a144bdd3,f6dd3ec061db4e3987629fe6b26e5cce,delivered,2017-04-26 10:53:06,2017-04-26 11:05:13,2017-05-04 14:35:00,2017-05-12 16:04:24,2017-05-15,1,credit_card,...,NaN,NaN,2017-05-13 00:00:00,2017-05-15 11:34:13,2017-04-26 10:53:06,-3 days +16:04:24,16 days 05:11:18,-3.0,16.0,Sudeste


In [ ]:
##Reduzindo a tabela, apenas com as colunas que serão utilizadas.
df_order_all = order_all[['order_id','order_status','order_item_id','order_approved_at','order_purchase_timestamp','order_date','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date','customer_unique_id','regiao','customer_state','customer_city','payment_type','payment_value','price','freight_value','product_category_name','review_score','dif_deliv_estimat_days','dif_deliv_compra_days']]
display(df_order_all.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 117329 entries, 0 to 117328
Data columns (total 21 columns):
 #   Column                         Non-Null Count   Dtype         
---  ------                         --------------   -----         
 0   order_id                       117329 non-null  object        
 1   order_status                   117329 non-null  object        
 2   order_item_id                  117329 non-null  int64         
 3   order_approved_at              117314 non-null  datetime64[ns]
 4   order_purchase_timestamp       117329 non-null  datetime64[ns]
 5   order_date                     117329 non-null  datetime64[ns]
 6   order_delivered_carrier_date   116094 non-null  datetime64[ns]
 7   order_delivered_customer_date  114858 non-null  datetime64[ns]
 8   order_estimated_delivery_date  117329 non-null  datetime64[ns]
 9   customer_unique_id             117329 non-null  object        
 10  regiao                         117329 non-null  object        
 11  

None

In [ ]:
#Remoção de registros duplicados df_order_all
original_rows = df_order_all.shape[0]
df_order_all.drop_duplicates(inplace=True)

print(f"Número original de linhas: {original_rows}")
print(f"Número de linhas após remover duplicatas: {df_order_all.shape[0]}")

Número original de linhas: 117329
Número de linhas após remover duplicatas: 116226


/tmp/ipykernel_11047/1768321591.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_order_all.drop_duplicates(inplace=True)


In [ ]:
#Avaliando todos os status do pedido presentes na base
display(df_order_all['order_status'].unique())

array(['delivered', 'invoiced', 'shipped', 'processing', 'canceled',
       'unavailable', 'approved'], dtype=object)

In [ ]:
#Lead time por regiao entre compra, aprovação, postagem e entrega
df_order_all['lead_time_compra_aprovacao'] = (df_order_all['order_purchase_timestamp']-df_order_all['order_approved_at'] ).dt.days
df_order_all['lead_time_aprovacao_postagem'] = (df_order_all['order_delivered_carrier_date']-df_order_all['order_approved_at'] ).dt.days
df_order_all['lead_time_postagem_entrega'] = (df_order_all['order_delivered_customer_date']-df_order_all['order_delivered_carrier_date'] ).dt.days

#Aplicação de filtro no Data Frame para que a base de analise tenha apenas pedidos entregues
df_order_all_deliv=df_order_all.where(df_order_all['order_status']=='delivered')
df_order_all_deliv=df_order_all.where(df_order_all['order_item_id']==1)
df_order_all_deliv=df_order_all_deliv.dropna()
df_order_all_deliv['anomes'] = df_order_all_deliv['order_purchase_timestamp'].dt.to_period('M')
df_order_all_deliv = df_order_all_deliv[(df_order_all_deliv['anomes'] >= pd.Period('2017-01', freq='M')) & (df_order_all_deliv['anomes'] <= pd.Period('2018-08', freq='M'))]
df_order_all_deliv.info()

/tmp/ipykernel_11047/580193947.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_order_all['lead_time_compra_aprovacao'] = (df_order_all['order_purchase_timestamp']-df_order_all['order_approved_at'] ).dt.days
/tmp/ipykernel_11047/580193947.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_order_all['lead_time_aprovacao_postagem'] = (df_order_all['order_delivered_carrier_date']-df_order_all['order_approved_at'] ).dt.days
/tmp/ipykernel_11047/580193947.py:4: SettingWithCopyWarning: 
A value is tryin

<class 'pandas.core.frame.DataFrame'>
Index: 98002 entries, 0 to 117328
Data columns (total 25 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       98002 non-null  object        
 1   order_status                   98002 non-null  object        
 2   order_item_id                  98002 non-null  float64       
 3   order_approved_at              98002 non-null  datetime64[ns]
 4   order_purchase_timestamp       98002 non-null  datetime64[ns]
 5   order_date                     98002 non-null  datetime64[ns]
 6   order_delivered_carrier_date   98002 non-null  datetime64[ns]
 7   order_delivered_customer_date  98002 non-null  datetime64[ns]
 8   order_estimated_delivery_date  98002 non-null  datetime64[ns]
 9   customer_unique_id             98002 non-null  object        
 10  regiao                         98002 non-null  object        
 11  customer_state     

In [ ]:
df_order_all_deliv.to_csv('/df_order_all_deliv3.csv', index=False)
print('DataFrame df_order_all_deliv exported to df_order_all_deliv.csv')

DataFrame df_order_all_deliv exported to df_order_all_deliv.csv


 Lead time entre compra, aprovação, postagem e entrega. • Correlação entre atrasos e review_score

### Correlação entre Atrasos e Review Score

Satisfação do Cliente • Distribuição de review_score e temas recorrentes (se text mining). • Drivers de satisfação: tempo de entrega, preço, categoria.  • Identificação de riscos de churn.

Oportunidades e Recomendação • Otimização de frete em rotas críticas. • Estratégias de pricing/cross-sell por categoria/região. • Priorizar sellers com alto NPS e baixo tempo de entrega.

Necessário cálcular o NPS